# 01 · Baseline: Text → Embedding Pipeline

Goal: understand exactly what the diffusion model "sees" when given a prompt.

1. Tokenize captions with the **CLIP ViT-L/14** tokenizer (the text encoder used by Stable Diffusion 1.5).
2. Produce **two** kinds of text embeddings:
   - `last_hidden_state` **(77 × 768)** – per-token conditioning fed to the UNet via cross-attention.
   - `text_embeds` **(768 → projected)** – pooled, L2-normalised vector in the shared CLIP image–text space.
3. Embed the matching images and measure **text→image retrieval (Recall@K)** and **CLIPScore** on our curated subset — this is our baseline alignment metric.
4. Confirm the SD pipeline's own text encoder produces identical conditioning.

Run `scripts/download_coco.py` and `scripts/preprocess.py` first.

In [ ]:
import json, random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer, CLIPTextModel

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
CLIP_NAME = "openai/clip-vit-large-patch14"
torch.manual_seed(0); random.seed(0)
print(DEVICE, torch.__version__)

## 1 · Load the curated dataset

In [ ]:
rows = [json.loads(l) for l in open(DATA / "metadata.jsonl", encoding="utf-8")]
print(f"{len(rows)} pairs | train={sum(r['split']=='train' for r in rows)} val={sum(r['split']=='val' for r in rows)}")
print(json.load(open(DATA / "stats.json")))

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, r in zip(axes, random.sample(rows, 4)):
    ax.imshow(Image.open(DATA / r["file_name"])); ax.axis("off")
    ax.set_title(r["text"][:60] + ("…" if len(r["text"]) > 60 else ""), fontsize=9, wrap=True)
plt.tight_layout()

## 2 · Tokenization

In [ ]:
tokenizer = CLIPTokenizer.from_pretrained(CLIP_NAME)
caption = rows[0]["text"]
enc = tokenizer(caption, padding="max_length", max_length=tokenizer.model_max_length,
                truncation=True, return_tensors="pt")
ids = enc.input_ids[0]
n_real = int(enc.attention_mask[0].sum())
print("caption :", caption)
print("tokens  :", tokenizer.convert_ids_to_tokens(ids[:n_real]))
print(f"length  : {n_real} real tokens, padded to {len(ids)}  (BOS={ids[0].item()}, EOS={tokenizer.eos_token_id})")

lens = [len(tokenizer(r["text"]).input_ids) for r in rows]
plt.figure(figsize=(6, 3)); plt.hist(lens, bins=30); plt.axvline(77, color="r", ls="--", label="CLIP limit")
plt.xlabel("tokens / caption"); plt.ylabel("count"); plt.legend(); plt.title("Caption token lengths");

## 3 · Text embeddings: per-token (UNet conditioning) vs pooled (CLIP space)

In [ ]:
clip = CLIPModel.from_pretrained(CLIP_NAME, torch_dtype=DTYPE).to(DEVICE).eval()
processor = CLIPProcessor.from_pretrained(CLIP_NAME)

with torch.no_grad():
    out = clip.text_model(input_ids=enc.input_ids.to(DEVICE))
    per_token = out.last_hidden_state                         # (1, 77, 768)  -> UNet cross-attention
    pooled = clip.text_projection(out.pooler_output)          # (1, 768)      -> shared image/text space
print("per-token conditioning:", tuple(per_token.shape))
print("pooled text embedding :", tuple(pooled.shape))

In [ ]:
@torch.no_grad()
def embed_texts(texts, bs=128):
    embs = []
    for i in range(0, len(texts), bs):
        t = tokenizer(texts[i:i+bs], padding=True, truncation=True, max_length=77, return_tensors="pt").to(DEVICE)
        embs.append(torch.nn.functional.normalize(clip.get_text_features(**t).float(), dim=-1).cpu())
    return torch.cat(embs)

@torch.no_grad()
def embed_images(paths, bs=64):
    embs = []
    for i in range(0, len(paths), bs):
        ims = [Image.open(p).convert("RGB") for p in paths[i:i+bs]]
        px = processor(images=ims, return_tensors="pt").pixel_values.to(DEVICE, DTYPE)
        embs.append(torch.nn.functional.normalize(clip.get_image_features(pixel_values=px).float(), dim=-1).cpu())
    return torch.cat(embs)

val = [r for r in rows if r["split"] == "val"]
T = embed_texts([r["text"] for r in val])
I = embed_images([DATA / r["file_name"] for r in val])
print("text:", tuple(T.shape), " image:", tuple(I.shape))

## 4 · Baseline alignment metrics (val split)

In [ ]:
sim = T @ I.T                                   # cosine similarity, (N_text, N_img)
gt = torch.arange(len(val))
ranks = (sim.argsort(dim=1, descending=True) == gt[:, None]).float().argmax(dim=1)
for k in (1, 5, 10):
    print(f"Text→Image Recall@{k:<2}: {(ranks < k).float().mean():.3f}")

clip_score = 100 * sim.diag().clamp(min=0)
print(f"CLIPScore (real pairs): mean={clip_score.mean():.2f}  std={clip_score.std():.2f}")
np.save(DATA / "val_text_emb.npy", T.numpy()); np.save(DATA / "val_image_emb.npy", I.numpy())

In [ ]:
n = 12
plt.figure(figsize=(6, 5))
plt.imshow(sim[:n, :n], cmap="viridis"); plt.colorbar(label="cosine sim")
plt.xlabel("image"); plt.ylabel("caption"); plt.title("Caption–image similarity (diagonal = true pairs)");

## 5 · Embedding space at a glance (PCA)

In [ ]:
from sklearn.decomposition import PCA
Z = PCA(2).fit_transform(torch.cat([T, I]).numpy())
plt.figure(figsize=(6, 5))
plt.scatter(*Z[:len(T)].T, s=6, alpha=.5, label="text"); plt.scatter(*Z[len(T):].T, s=6, alpha=.5, label="image")
plt.legend(); plt.title("CLIP embeddings – modality gap visible as two clusters");

## 6 · Same conditioning inside Stable Diffusion
The SD 1.5 pipeline ships the same CLIP ViT-L/14 text encoder. We load only that sub-module (≈ 500 MB)
and confirm it yields identical per-token embeddings — this is the exact tensor passed to the UNet.

In [ ]:
SD_NAME = "stable-diffusion-v1-5/stable-diffusion-v1-5"
sd_tok = CLIPTokenizer.from_pretrained(SD_NAME, subfolder="tokenizer")
sd_te = CLIPTextModel.from_pretrained(SD_NAME, subfolder="text_encoder", torch_dtype=DTYPE).to(DEVICE).eval()

prompt = "A red double-decker bus driving down a rainy London street."
ids_sd = sd_tok(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to(DEVICE)
ids_clip = tokenizer(prompt, padding="max_length", max_length=77, truncation=True, return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad():
    cond_sd = sd_te(ids_sd).last_hidden_state
    cond_clip = clip.text_model(input_ids=ids_clip).last_hidden_state
print("SD conditioning:", tuple(cond_sd.shape))
print("max |diff| vs CLIPModel:", (cond_sd - cond_clip).abs().max().item())

## Takeaways / next steps
- Captions fit the 77-token window after filtering; the UNet receives a **77×768** tensor per prompt.
- Recall@K and CLIPScore on real pairs give an **upper-bound reference** for scoring generated images later.
- Next: run `scripts/generate_samples.py` (5 prompt→image runs) and compare generated-image CLIPScore against this baseline; then fine-tune (LoRA) on the train split.